In [50]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')


In [51]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATA")
    print("=" * 60)

    pd.set_option('display.float_format', lambda x: f'{x:.2f}')
    
    df = pd.read_csv(file_path)
    print("Shape of Dataset:")
    print(df.shape)
    print("\nChecking for Missing Value:")
    print(df.isnull().sum())
    print("\nFirst 5 Row:")
    print(df.head())
    print("\nDescriptive Statistics:")
    print(df.describe())
    print("\nDataset Info:")
    print(df.info())
    print("\nCondition Distribution:")
    print(df["Condition"].value_counts())
    print("\nTransmission Distribution:")
    print(df["Transmission"].value_counts())

    return df


In [52]:
#le.fit_transform - rearrange the unique value and transform into numeric form using index
#le.classes_ returns the encoded value

def preprocessed_data(df):
    print("=" * 60)
    print("PREPROCESSING DATA")
    print("=" * 60)

    df_preprocessed = df.copy()
    df_preprocessed = df.dropna()

    label_encoder = {}
    df_columns = ["Make", "Model", "Condition", "Transmission"]
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder[col] = le
        print(f"\n{col} encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")
    print("Preprocessed Data Shape", df_preprocessed.shape)

    return df_preprocessed, label_encoder



In [53]:
def featured_data(df_preprocessed):
    print("\n" + "=" * 60)
    print("FEATURES DATASET")
    print("=" * 60)

    feature_columns = ["Year", "Make_encoded", "Model_encoded", "Transmission_encoded", "Condition_encoded"]
    target = ["Price"]

    X = df_preprocessed[feature_columns]
    y = df_preprocessed[target]

    print("\nFeature Shape:", df_preprocessed[feature_columns].shape)
    print("\nTarget Shape:", df_preprocessed[target].shape)
    print("\nFeature Columns:", feature_columns)

    return X, y, feature_columns


In [54]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITING DATASET")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining Set Size:", X_train.shape[0])
    print("\nTesting Set Size:", X_test.shape[0])
    print("\nTraining Set Range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    
    print("\nTesting Set Range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))
    return X_train, X_test, y_train, y_test

In [55]:
def scale_features(X_train, X_test):
    print("\n" + "=" * 60)
    print("SCALE FEATURE DATASET")
    print("=" * 60)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\nFeatures Scaled Successful")
    print("\nTrain Scaled Features:", X_train_scaled.shape)
    print("\nTest Scaled Features:", X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, scaler

In [56]:
def train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)

    # Initializing
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

#zip is added to enable iteration on 2 objects
#.ravel transform the objects iterated objects into single dimentional array
#intercept is the baseline if all the features is 0
    
    for feature, coef in zip(feature_columns, model.coef_.ravel()):
        print(f"   {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model
    

In [57]:
def rm_train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("RANDOM FOREST TRAINING MODEL")
    print("=" * 60)

    rm_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rm_model.fit(X_train_scaled, y_train)

    print("\nModel Trained Successfully")

    feature_importance = sorted(zip(feature_columns, rm_model.feature_importances_), 
                                key=lambda x: x[1], reverse=True)
    for feature, importance in feature_importance:
        print(f" {feature}: {float(importance)}")
 
    return rm_model


In [58]:
def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    #calculate metrics

    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        


    return metrics

In [59]:
def rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("RANDOM FOREST EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction
    y_train_pred = rm_model.predict(X_train_scaled)
    y_test_pred = rm_model.predict(X_test_scaled)

    #calculate metrics

    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("RANDOM MODEL MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    rm_metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        
    return rm_metrics

In [60]:
def save_model_artifact(model, scaler, label_encoder, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "car_price_prediction.pkl")
    print("Car Price Prediction Model Saved")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler Features Saved")

    joblib.dump(label_encoder, "label_encoder.pkl")
    print("Label Feature Encoder Saved")

    joblib.dump(feature_columns, "feature_columns.pkl")
    print("Feature Columns Saved")
    

In [61]:
def save_rm_model_artifact(rm_model):
    print("\n" + "=" * 60)
    print("SAVING RANDOM FOREST MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_car_price_prediction.pkl")
    print("Car Price Prediction Model Saved")

In [62]:
def predict_car_price(year, make, model_name, transmission, condition):
    model = joblib.load("car_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")

    try:
        make_encoded = label_encoder['Make'].transform([make])[0]
        model_encoded = label_encoder['Model'].transform([model_name])[0]
        condition_encoded = label_encoder['Condition'].transform([condition])[0]
        transmission_encoded = label_encoder['Transmission'].transform([transmission])[0]
    except ValueError as e:
        return f"Unknown Category {e}"

    features_dict = {
        "Year": year,
        "Make_encoded": make_encoded,
        "Model_encoded": model_encoded,
        "Transmission_encoded": transmission_encoded,
        "Condition_encoded": condition_encoded 
    }

    features = np.array([[features_dict[col] for col in features_dict]])

    feature_scale = scaler.transform(features)

    predicted_price = model.predict(feature_scale)[0].item()

    return predicted_price

In [63]:
def rm_model_predict_car_price(year, make, model_name, transmission, condition):
    rm_model = joblib.load("rm_model_car_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")

    try:
        make_encoded = label_encoder['Make'].transform([make])[0]
        model_encoded = label_encoder['Model'].transform([model_name])[0]
        condition_encoded = label_encoder['Condition'].transform([condition])[0]
        transmission_encoded = label_encoder['Transmission'].transform([transmission])[0]
    except ValueError as e:
        return f"Unknown Category {e}"

    features_dict = {
        "Year": year,
        "Make_encoded": make_encoded,
        "Model_encoded": model_encoded,
        "Transmission_encoded": transmission_encoded,
        "Condition_encoded": condition_encoded 
    }

    features = np.array([[features_dict[col] for col in features_dict]])

    feature_scale = scaler.transform(features)

    predicted_price = rm_model.predict(feature_scale)[0].item()

    return predicted_price

In [64]:
def test_prediction():
    print("\n" + "=" * 60)
    print("TEST CAR PREDICTION")
    print("=" * 60)

    #EXAMPLE 1
    price1 = predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    print("\n 2015 Toyota Camry (Local Used Automatic)")
    print(f" ₦{float(price1):,.2f}")

    #EXAMPLE 1
    price2 = predict_car_price(2012, "Lexus", "Rx 350", "Automatic", "Foreign Used")
    print("\n 2012 Lexus Rx 350 (Foreign Used Automatic)")
    print(f" ₦{float(price2):,.2f}")

    #EXAMPLE 1
    price3 = predict_car_price(2010, "Honda", "Accord", "Automatic", "Local Used")
    print("\n 2010 Honda Accord (Local Used Automatic)")
    print(f" ₦{float(price3):,.2f}")
    

In [65]:
def rm_model_test_prediction():
    print("\n" + "=" * 60)
    print("RANDOM FOREST TEST CAR PREDICTION")
    print("=" * 60)

    #EXAMPLE 1
    price1 = rm_predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    print("\n 2015 Toyota Camry (Local Used Automatic)")
    print(f" ₦{float(price1):,.2f}")

    #EXAMPLE 1
    price2 = rm_predict_car_price(2012, "Lexus", "Rx 350", "Automatic", "Foreign Used")
    print("\n 2012 Lexus Rx 350 (Foreign Used Automatic)")
    print(f" ₦{float(price2):,.2f}")

    #EXAMPLE 1
    price3 = rm_predict_car_price(2010, "Honda", "Accord", "Automatic", "Local Used")
    print("\n 2010 Honda Accord (Local Used Automatic)")
    print(f" ₦{float(price3):,.2f}")
    

In [66]:
def main():
    file_path = "data/cleaned_jiji_autos.csv"

    df = load_and_explore_dataset(file_path)

    df_preprocessed, label_encoder = preprocessed_data(df)

    X, y, feature_columns = featured_data(df_preprocessed)

    X_train, X_test, y_train, y_test = split_data(X, y)

    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

    model = train_model(X_train_scaled, y_train, feature_columns)

    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)    

    rm_model = rm_train_model(X_train_scaled, y_train, feature_columns)

    rm_metrics = rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test)

    save_model_artifact(model, scaler, label_encoder, feature_columns)

    save_rm_model_artifact(rm_model)

    test_prediction()

    rm_model_test_prediction()

In [67]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATA
Shape of Dataset:
(1790, 7)

Checking for Missing Value:
Title           0
Make            0
Model           0
Year            0
Condition       0
Transmission    0
Price           0
dtype: int64

First 5 Row:
                                               Title           Make  \
0                             Toyota Camry 2007 Gray         Toyota   
1                                Lexus ES 2014 Black          Lexus   
2  Mercedes-Benz M Class ML350 4MATIC 4dr SUV AWD...  Mercedes-Benz   
3                         Hyundai Santa Fe 2013 Blue        Hyundai   
4          Toyota Corolla Sedan Automatic 2003 Black         Toyota   

           Model  Year     Condition Transmission       Price  
0          Camry  2007    Local Used    Automatic  6960937.00  
1             Es  2014    Local Used    Automatic 13500000.00  
2        M Class  2014  Foreign Used    Automatic 17300000.00  
3       Santa Fe  2013    Local Used    Automatic 10000000.00  
4  Corolla Sedan  200